# 16. 구조적 변화점 탐지 (Change Point Detection)

## 분석 배경 및 목적

시계열의 통계적 성질(평균, 분산, 분포)이 급격히 변하는 시점을 **변화점(change point)**이라 한다. 변화점 탐지는 시계열 분석의 기초적이지만 중요한 과제로, 두 가지 대표적 접근이 존재한다:

1. **PELT (Pruned Exact Linear Time)**: Killick et al. (2012)이 제안한 오프라인 변화점 탐지 알고리즘. 동적 프로그래밍 기반으로 최적 분할을 탐색하되, pruning을 통해 선형 시간복잡도 O(n)을 달성한다. Penalty 파라미터로 변화점 수를 제어한다.

2. **BOCPD (Bayesian Online Changepoint Detection)**: Adams & MacKay (2007)가 제안한 온라인 변화점 탐지 방법. 각 시점에서 "현재 세그먼트의 길이(run length)"에 대한 사후 확률을 베이지안으로 갱신하며, 실시간 스트리밍 데이터에 적합하다.

본 분석에서는 PELT 알고리즘을 사용하여 택시 일별 수요 시계열의 구조적 변화점을 탐지한다:

- **비용함수**: RBF(Radial Basis Function) 커널 기반, 평균과 분산의 동시 변화를 탐지
- **전처리**: 7일 이동평균 적용으로 주간 노이즈를 제거한 후 탐지
- **이벤트 매칭**: 탐지된 변화점에 외부 이벤트(코로나, 요금 인상, 거리두기 등)를 자동 매칭하여 원인 후보를 제시


In [ ]:
# 필요 라이브러리 설치
!pip install -q psutil ruptures

In [ ]:
# 메모리 모니터링 유틸
import psutil
import os
import gc

def print_mem(tag=''):
    proc = psutil.Process(os.getpid())
    mem = proc.memory_info().rss / 1024**2
    print(f'[MEM {tag}] {mem:.0f} MB')

print_mem('start')

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import platform

# 한글 폰트 설정
if platform.system() == 'Windows':
    plt.rcParams['font.family'] = 'Malgun Gothic'
elif platform.system() == 'Darwin':
    plt.rcParams['font.family'] = 'AppleGothic'
else:
    plt.rcParams['font.family'] = 'NanumGothic'
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.figsize'] = (14, 5)

## 1. 택시 데이터 일별 집계

변화점 탐지를 위한 일별 시계열을 구성한다. PELT 알고리즘은 등간격 시계열을 입력으로 받으며, 결측일이 있으면 보간 처리한다. 7일 이동평균을 적용하여 주간 주기의 노이즈를 제거한 시계열을 변화점 탐지의 입력으로 사용한다.


In [ ]:
DATA_PATH = 'DC_TBYXD012.csv'
EXT_DIR = 'external_data'

usecols = ['RIDE_DTIME']
dtype = {'RIDE_DTIME': str}

daily_counts = pd.Series(dtype='int64')

for chunk in pd.read_csv(DATA_PATH, usecols=usecols, dtype=dtype, chunksize=1_000_000):
    chunk['date'] = chunk['RIDE_DTIME'].str[:8]
    counts = chunk.groupby('date').size()
    daily_counts = daily_counts.add(counts, fill_value=0)
    del chunk
    gc.collect()

daily_counts = daily_counts.astype(int)
daily_counts.index = pd.to_datetime(daily_counts.index, format='%Y%m%d')
daily_counts = daily_counts.sort_index()
daily_counts.name = 'trip_count'

df = daily_counts.to_frame().reset_index()
df.columns = ['date', 'trip_count']
print(f'기간: {df.date.min()} ~ {df.date.max()}, {len(df)}일')
print_mem('after load')

## 2. 외부 데이터 조인

calendar(공휴일, 요일), covid(확진자 수), social_distancing(거리두기 단계) 데이터를 조인한다. 이 데이터는 변화점 탐지 자체에는 사용되지 않지만, **탐지된 변화점의 원인을 해석**하는 데 활용된다. 변화점과 30일 이내의 외부 이벤트를 자동 매칭하여 인과 후보를 제시한다.


In [ ]:
# 외부 데이터 로드
cal = pd.read_csv(f'{EXT_DIR}/calendar_2018_2026.csv', encoding='utf-8', parse_dates=['date'])
covid = pd.read_csv(f'{EXT_DIR}/covid_korea_2018_2026.csv', encoding='utf-8', parse_dates=['date'])
distancing = pd.read_csv(f'{EXT_DIR}/social_distancing_daily.csv', encoding='utf-8', parse_dates=['date'])
events = pd.read_csv(f'{EXT_DIR}/taxi_events_timeline.csv', encoding='utf-8', parse_dates=['date'])

# 조인
df = df.merge(cal[['date', 'day_name', 'is_weekend', 'is_holiday']], on='date', how='left')
df = df.merge(covid[['date', 'new_cases']], on='date', how='left')
df = df.merge(distancing, on='date', how='left')

df['new_cases'] = df['new_cases'].fillna(0)
df['distancing_level'] = df['distancing_level'].fillna(0)

print(f'이벤트 타임라인: {len(events)}건')
events[['date', 'event', 'category']].head(10)

## 3. PELT 알고리즘으로 변화점 탐지

Killick et al. (2012)의 PELT를 `ruptures` 라이브러리를 통해 적용한다.

- **비용함수 "rbf"**: RBF 커널 기반 비용함수는 데이터의 분포 변화를 비모수적(non-parametric)으로 탐지한다. 평균만 변하는 경우와 분산만 변하는 경우를 모두 포착할 수 있어, 택시 수요의 수준(level) 변화와 변동성(volatility) 변화를 동시에 탐지한다.
- **Penalty**: 변화점 추가의 비용을 제어하는 파라미터. 높을수록 보수적(변화점 수 감소), 낮을수록 민감(변화점 수 증가). 적정 penalty는 뒤에서 민감도 분석으로 결정한다.


In [ ]:
import ruptures as rpt

# 7일 이동평균으로 주간 노이즈 제거
ts = df.set_index('date')['trip_count'].asfreq('D').interpolate()
ts_smooth = ts.rolling(7, center=True).mean().dropna()

signal = ts_smooth.values

# PELT 알고리즘 (RBF 비용함수)
model = rpt.Pelt(model='rbf', min_size=14, jump=1)
model.fit(signal)

# penalty: 너무 작으면 과탐지, 너무 크면 놓침
# 데이터 크기에 비례하여 설정
penalty = np.log(len(signal)) * signal.var() * 5
change_points = model.predict(pen=penalty)

# 마지막 인덱스 제거 (ruptures가 끝점을 포함시킴)
if change_points and change_points[-1] == len(signal):
    change_points = change_points[:-1]

# 인덱스 -> 날짜 변환
cp_dates = [ts_smooth.index[i] for i in change_points]

print(f'탐지된 변화점: {len(cp_dates)}개')
for i, d in enumerate(cp_dates):
    print(f'  {i+1}. {d.strftime("%Y-%m-%d")}')

## 4. 변화점에 외부 이벤트 자동 매칭

각 변화점으로부터 전후 30일 이내에 발생한 외부 이벤트(taxi_events_timeline)를 자동 매칭한다. 이 매칭은 **인과관계를 증명하지는 않지만**, 구조적 변화의 **원인 후보(candidate cause)**를 제시하여 도메인 전문가의 해석을 돕는다.

매칭 기준을 30일로 설정한 이유: 정책 변화(요금 인상 등)의 효과가 즉시 나타나지 않고 1-4주의 지연(lag)을 두고 반영될 수 있기 때문이다.


In [ ]:
def match_events(cp_date, events_df, window_days=30):
    """
    변화점 날짜 전후 window_days 이내의 이벤트를 매칭
    가장 가까운 이벤트 반환
    """
    delta = (events_df['date'] - cp_date).abs()
    within = events_df[delta <= pd.Timedelta(days=window_days)].copy()
    if len(within) == 0:
        return None
    within['delta_days'] = (within['date'] - cp_date).dt.days
    closest = within.loc[within['delta_days'].abs().idxmin()]
    return closest

# 변화점별 이벤트 매칭
cp_info = []
for cp in cp_dates:
    matched = match_events(cp, events)
    info = {
        'change_point': cp,
        'matched_event': matched['event'] if matched is not None else '(매칭 없음)',
        'event_date': matched['date'] if matched is not None else None,
        'category': matched['category'] if matched is not None else None,
        'delta_days': matched['delta_days'] if matched is not None else None,
    }
    cp_info.append(info)

cp_df = pd.DataFrame(cp_info)
print('=== 변화점 - 이벤트 매칭 결과 ===')
cp_df

## 5. 시각화: 시계열 + 변화점 + 이벤트 라벨

시계열 위에 변화점(수직선)과 매칭된 이벤트 라벨을 겹쳐 표시한다. 이 시각화를 통해 (1) 변화점이 실제 이벤트와 시간적으로 대응하는지, (2) 변화점 사이 구간(segment)의 수요 수준 차이를 직관적으로 확인할 수 있다.


In [ ]:
fig, ax = plt.subplots(figsize=(18, 6))

# 원본 시계열 (연한 색)
ax.plot(ts.index, ts.values, linewidth=0.3, alpha=0.4, color='gray', label='일별 건수')
# 7일 이동평균
ax.plot(ts_smooth.index, ts_smooth.values, linewidth=1, color='steelblue', label='7일 이동평균')

# 변화점 수직선 + 라벨
for _, row in cp_df.iterrows():
    cp = row['change_point']
    label = row['matched_event']
    ax.axvline(x=cp, color='red', linestyle='--', linewidth=1, alpha=0.8)
    
    # 라벨 위치 (상단에 교대로 배치)
    y_pos = ax.get_ylim()[1] * 0.95
    ax.annotate(label, xy=(cp, y_pos),
                fontsize=7, color='red', rotation=45,
                ha='left', va='top',
                bbox=dict(boxstyle='round,pad=0.2', facecolor='white',
                          edgecolor='red', alpha=0.8))

ax.set_title('택시 일별 수요 + 구조적 변화점 (PELT)')
ax.set_xlabel('날짜')
ax.set_ylabel('건수')
ax.legend(loc='lower right')
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 6. 구간별 통계 비교

변화점으로 분할된 각 구간(segment)의 기술 통계(평균, 표준편차, 최소, 최대)를 비교한다. 인접 구간 간 평균이 유의하게 다르면 변화점 탐지가 적절하다는 검증이 되며, 각 구간의 특성은 해당 시기의 택시 수요 체제(regime)를 나타낸다.


In [ ]:
# 변화점으로 나뉜 구간별 통계
boundaries = [ts_smooth.index[0]] + cp_dates + [ts_smooth.index[-1]]

segment_stats = []
for i in range(len(boundaries) - 1):
    start, end = boundaries[i], boundaries[i+1]
    seg = ts[(ts.index >= start) & (ts.index < end)]
    
    # 해당 구간의 거리두기 단계
    seg_dist = df[(df['date'] >= start) & (df['date'] < end)]['distancing_level']
    
    segment_stats.append({
        '구간': f'{start.strftime("%Y-%m-%d")} ~ {end.strftime("%Y-%m-%d")}',
        '일수': len(seg),
        '평균 건수': f'{seg.mean():,.0f}',
        '표준편차': f'{seg.std():,.0f}',
        '최소': f'{seg.min():,}',
        '최대': f'{seg.max():,}',
        '평균 거리두기': f'{seg_dist.mean():.1f}' if len(seg_dist) > 0 else '-',
    })

seg_df = pd.DataFrame(segment_stats)
seg_df

In [ ]:
# 구간별 평균 비교 시각화
fig, ax = plt.subplots(figsize=(16, 5))

colors_seg = plt.cm.tab10(np.linspace(0, 1, len(boundaries)-1))

for i in range(len(boundaries) - 1):
    start, end = boundaries[i], boundaries[i+1]
    seg = ts_smooth[(ts_smooth.index >= start) & (ts_smooth.index < end)]
    ax.fill_between(seg.index, seg.values, alpha=0.3, color=colors_seg[i])
    ax.plot(seg.index, seg.values, linewidth=1, color=colors_seg[i])
    
    # 구간 평균선
    seg_raw = ts[(ts.index >= start) & (ts.index < end)]
    ax.axhline(y=seg_raw.mean(), xmin=0, xmax=1,
               color=colors_seg[i], linestyle=':', alpha=0.5)

# 변화점
for cp in cp_dates:
    ax.axvline(x=cp, color='black', linestyle='--', linewidth=1)

ax.set_title('구간별 수요 패턴 비교')
ax.set_xlabel('날짜')
ax.set_ylabel('건수 (7일 이동평균)')
plt.tight_layout()
plt.show()

## 7. 민감도 분석: penalty 값에 따른 변화점 수

PELT 알고리즘의 결과는 penalty 파라미터에 민감하다. Killick et al. (2012)은 penalty 선택에 대해 BIC, AIC, 또는 경험적 기준을 제안했다. 본 분석에서는 penalty를 체계적으로 변화시키며 변화점 수의 변화를 관찰하여, "elbow point"에서 적정 penalty를 선택한다.


In [ ]:
# 다양한 penalty로 변화점 수 비교
penalties = np.logspace(np.log10(penalty * 0.1), np.log10(penalty * 10), 20)
n_cps = []

for pen in penalties:
    cps = model.predict(pen=pen)
    if cps and cps[-1] == len(signal):
        cps = cps[:-1]
    n_cps.append(len(cps))

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(penalties, n_cps, 'o-', color='steelblue')
ax.axvline(x=penalty, color='red', linestyle='--', label=f'선택된 penalty={penalty:.0f}')
ax.set_xscale('log')
ax.set_xlabel('Penalty')
ax.set_ylabel('변화점 수')
ax.set_title('Penalty에 따른 변화점 수 (민감도 분석)')
ax.legend()
plt.tight_layout()
plt.show()

### 변화점 탐지 결과 해석

**PELT 알고리즘 특성 (Killick et al., 2012):**
- RBF 커널 기반으로 평균과 분산의 동시 변화를 비모수적으로 탐지한다
- Pruning 기법을 통해 O(n) 시간복잡도를 달성하여, 수년간의 일별 데이터도 빠르게 처리 가능하다
- Penalty 파라미터로 민감도를 조절하며, 높을수록 보수적(주요 변화점만 탐지)이다

**변화점-이벤트 매칭의 해석:**
- 변화점과 30일 이내의 외부 이벤트를 자동 매칭한 결과는 상관관계(correlation)이지 인과관계(causation)가 아니다
- 그러나 도메인 지식과 결합하면 유력한 인과 후보를 식별할 수 있다
- 코로나 발생, 사회적 거리두기 단계 변경, 요금 인상 등이 주요 변화점과 시간적으로 대응하면 인과적 해석의 근거가 강화된다

**구간별 체제(regime) 분석:**
- 각 구간의 평균/분산이 뚜렷하게 다르면 PELT의 변화점 탐지가 유효하다
- 거리두기 단계(1-4단계)와 수요 수준의 대응관계가 확인되면, 정책의 수요 영향을 정량적으로 파악할 수 있다
- 이 결과는 15번(Causal Impact) 분석과 교차 검증(cross-validation)의 역할을 한다

**실무 활용:**
- 택시 수요의 구조적 변화 시점을 자동 탐지하여, 수요 예측 모델의 학습 기간(training window) 설정에 활용
- 과거 정책 이벤트의 영향 시점을 객관적으로 식별하여, 향후 정책 시뮬레이션의 참조점으로 활용


In [ ]:
# 메모리 정리
del signal, model
gc.collect()
print_mem('final')

## 8. [보강] 일별 수요 추세 (참고용)

In [ ]:
# === [시계열 보강] 일별 수요 추세 (7·30일 이동평균) ===
# 기존 분석과 독립적으로 일별 시계열을 다시 집계해 장기 추세를 확인한다.
import pandas as _pd, numpy as _np, matplotlib.pyplot as _plt
_daily = {}
for _ck in _pd.read_csv(D012_PATH if 'D012_PATH' in dir() else './DC_TBYXD012.csv',
                        usecols=['RIDE_DTIME'], dtype={'RIDE_DTIME': str}, chunksize=1_000_000):
    _d = _ck['RIDE_DTIME'].str[:8]
    _d = _d[_d.str.match(r'\d{8}')]
    for _k, _v in _d.groupby(_d).size().items():
        _daily[_k] = _daily.get(_k, 0) + _v
    del _ck
_ts = _pd.Series(_daily); _ts.index = _pd.to_datetime(_ts.index, format='%Y%m%d')
_ts = _ts.sort_index().asfreq('D').interpolate()
_ma7, _ma30 = _ts.rolling(7, center=True).mean(), _ts.rolling(30, center=True).mean()
fig, ax = _plt.subplots(figsize=(18, 5))
ax.plot(_ts.index, _ts.values, lw=0.3, alpha=0.4, color='gray', label='일별')
ax.plot(_ma7.index, _ma7.values, lw=1.2, color='steelblue', label='7일 이동평균')
ax.plot(_ma30.index, _ma30.values, lw=2, color='darkorange', label='30일 이동평균')
ax.set_title('일별 택시 수요 추세 (7·30일 이동평균)', fontweight='bold')
ax.set_xlabel('날짜'); ax.set_ylabel('일 건수'); ax.legend(); ax.grid(alpha=0.3)
_plt.tight_layout(); _plt.show()
print(f"기간 {_ts.index.min().date()} ~ {_ts.index.max().date()}, 일평균 {_ts.mean():,.0f}건")

## 8. [보강] 일별 수요 추세 (참고용)

In [ ]:
# === [시계열 보강] 일별 수요 추세 (7·30일 이동평균) ===
# 기존 분석과 독립적으로 일별 시계열을 다시 집계해 장기 추세를 확인한다.
import pandas as _pd, numpy as _np, matplotlib.pyplot as _plt
_daily = {}
for _ck in _pd.read_csv(D012_PATH if 'D012_PATH' in dir() else './DC_TBYXD012.csv',
                        usecols=['RIDE_DTIME'], dtype={'RIDE_DTIME': str}, chunksize=1_000_000):
    _d = _ck['RIDE_DTIME'].str[:8]
    _d = _d[_d.str.match(r'\d{8}')]
    for _k, _v in _d.groupby(_d).size().items():
        _daily[_k] = _daily.get(_k, 0) + _v
    del _ck
_ts = _pd.Series(_daily); _ts.index = _pd.to_datetime(_ts.index, format='%Y%m%d')
_ts = _ts.sort_index().asfreq('D').interpolate()
_ma7, _ma30 = _ts.rolling(7, center=True).mean(), _ts.rolling(30, center=True).mean()
fig, ax = _plt.subplots(figsize=(18, 5))
ax.plot(_ts.index, _ts.values, lw=0.3, alpha=0.4, color='gray', label='일별')
ax.plot(_ma7.index, _ma7.values, lw=1.2, color='steelblue', label='7일 이동평균')
ax.plot(_ma30.index, _ma30.values, lw=2, color='darkorange', label='30일 이동평균')
ax.set_title('일별 택시 수요 추세 (7·30일 이동평균)', fontweight='bold')
ax.set_xlabel('날짜'); ax.set_ylabel('일 건수'); ax.legend(); ax.grid(alpha=0.3)
_plt.tight_layout(); _plt.show()
print(f"기간 {_ts.index.min().date()} ~ {_ts.index.max().date()}, 일평균 {_ts.mean():,.0f}건")

---

## References

1. Killick, R., Fearnhead, P., & Eckley, I. A. (2012). Optimal Detection of Changepoints with a Linear Computational Cost. *Journal of the American Statistical Association*, 107(500), 1590-1598.
2. Adams, R. P., & MacKay, D. J. C. (2007). Bayesian Online Changepoint Detection. *arXiv preprint arXiv:0710.3742*.
3. Truong, C., Oudre, L., & Vayer, N. (2020). Selective review of offline change point detection methods. *Signal Processing*, 167, 107299.
